# SAE known-entity feature — correcting a hallucination

Replicates the steering experiment from **"Do I Know This Entity? Knowledge Awareness and Hallucinations in Language Models"** ([arXiv:2411.14257](https://arxiv.org/abs/2411.14257)) on Gemma-2-9B-it.

The question smuggles in a false premise (LeBron James won his first MVP in 2009, not 2006), and the baseline hallucinates: it accepts the wrong year and answers as if the premise were true. A "known entity" direction, read directly from the Gemma Scope SAE decoder (layer 31, width 16k, feature 88) and added at the last prompt position, restores the model's knowledge awareness: at moderate scale it corrects the year, and at high scale it rejects the false premise outright.


In [1]:
import os

from vllm import LLM, SamplingParams
import easysteer.vectors as vec
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

MODEL = "/home/xhl/huggingface_models/google/gemma-2-9b-it"  # or google/gemma-2-9b-it

llm = LLM(
    model=MODEL,
    enable_steer_vector=True,
    enforce_eager=True,
)

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


INFO 08-04 01:52:40 [api_utils.py:273] non-default args: {'disable_log_stats': True, 'enforce_eager': True, 'enable_steer_vector': True, 'model': '/home/xhl/huggingface_models/google/gemma-2-9b-it'}


INFO 08-04 01:52:40 [model.py:623] Resolved architecture: Gemma2ForCausalLM


INFO 08-04 01:52:40 [model.py:1788] Using max model len 8192


...

(EngineCore pid=4087104) 

WARNING 08-04 01:56:08 [vllm.py:1249] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


(EngineCore pid=4087104) 

(EngineCore pid=4087104) 

INFO 08-04 01:56:08 [vllm.py:1428] Cudagraph is disabled under eager mode


In [2]:
from transformers import AutoTokenizer

# False premise: LeBron James won his first MVP in 2009, not 2006.
messages = [
    {"role": "user", "content": "Who was the head coach of the Cleveland Cavaliers when LeBron James won his first MVP in 2006?"},
]
tokenizer = AutoTokenizer.from_pretrained(MODEL)
example = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
params = SamplingParams(temperature=0, max_tokens=256, skip_special_tokens=False)

baseline = llm.generate(example, params)
print("=====Baseline=====")
print(baseline[0].outputs[0].text)

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 18.11it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it, est. speed input: 19.06 toks/s, output: 16.75 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it, est. speed input: 19.06 toks/s, output: 16.75 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it, est. speed input: 19.06 toks/s, output: 16.75 toks/s]

=====Baseline=====
The head coach of the Cleveland Cavaliers when LeBron James won his first MVP in 2006 was **Mike Brown**. 
<end_of_turn>


In [3]:
import numpy as np
import torch

# Gemma Scope SAE for the layer-31 residual stream (width 16k).
SAE_PARAMS = "/home/xhl/huggingface_models/google/gemma-scope-9b-it-res/layer_31/width_16k/average_l0_76/params.npz"  # google/gemma-scope-9b-it-res

data = np.load(SAE_PARAMS)
W_dec = data["W_dec"]  # (16384, 3584): one decoder row per SAE feature
# The paper identifies features 88 and 5038 as the strongest
# "known entity" directions.
feature = 88  # also try 5038
torch.save(W_dec[feature, :], "james.pt")

In [4]:
# Larger scale pushes harder toward "known entity": α=500 corrects
# the year, α=2000 rejects the false premise outright.
for scale in (500, 2000):
    steering = SteeringSpec(vectors=[
        VectorSpec(
            data=vec.from_pt_direction("james.pt", layers=[31]),
            scale=scale,
            layers=[31],
            apply=ApplySpec(phases=["prompt"], positions=[-1]),
        ),
    ])
    steered = llm.generate(example, params, steering=steering)
    print(f"=====α {scale}=====")
    print(steered[0].outputs[0].text)

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 198.99it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.40s/it, est. speed input: 9.70 toks/s, output: 16.46 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.40s/it, est. speed input: 9.70 toks/s, output: 16.46 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.81s/it, est. speed input: 8.66 toks/s, output: 17.07 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.81s/it, est. speed input: 8.66 toks/s, output: 17.07 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.81s/it, est. speed input: 8.66 toks/s, output: 17.07 toks/s]

=====α 2000=====
Unfortunately, this is a bit of a trick question! 

LeBron James won his first MVP award in **2009**, not 2006. 

The head coach of the Cleveland Cavaliers when LeBron won his first MVP in 2009 was **Mike Brown**. 
<end_of_turn>
